In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

api_key = os.getenv("API_KEY")


In [11]:
import uuid

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.config import get_store
from langgraph.store.memory import InMemoryStore


model = ChatOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-4o-mini"
)

store = InMemoryStore();


# інструмент для додавання завдань

@tool
def add_task(task_description: str) -> str:
    """Додає нове завдання до списку з унікальним ID."""

    if not task_description.strip():
        return "Помилка: опис завдання не може бути порожнім."

    store = get_store()
    task_id = str(uuid.uuid4())

    store.put(
        ("tasks",),
        task_id,
        {"description": task_description.strip()}
    )

    return (
        f'Завдання "{task_description}" '
        f'було успішно додано. ID: {task_id}'
    )


# інструмент для перегляду всіх завдань
@tool
def list_tasks() -> str:
    """
    Показує всі збережені завдання з номерами та ID.
    """

    store = get_store()
    items = store.search(("tasks",))

    tasks = []
    for index, item in enumerate(items, start=1):
        tasks.append(
            f"{index}. {item.value['description']} "
            f"(ID: {item.key})"
        )


    if not tasks:
        return "Список завдань порожній."

    return "\n".join(tasks)


# інструмент для видалення завдання
@tool
def delete_task(task_id: str) -> str:
    """Видаляє завдання за повним UUID.

    Якщо відома лише назва, спочатку виклич list_tasks,
    знайди UUID і передай його цьому інструменту.
    """

    store = get_store()

    task = store.get(("tasks",), task_id)

    if task is None:
        return f"Завдання з ID {task_id} не знайдено."

    store.delete(("tasks",), task_id)

    return f'Завдання "{task.value["description"]}" було видалено.'

# 5. System prompt
SYSTEM_PROMPT = """
    
    Ти — TODO-агент для керування списком справ.
    Завжди відповідай українською мовою.

    Доступні інструменти:
    - add_task — додати завдання
    - list_tasks — переглянути завдання та знайти їхні ID
    - delete_task — видалити завдання за ID

    ОБОВ'ЯЗКОВІ ПРАВИЛА:

    1. Якщо користувач просить додати завдання:
    виклич add_task.

    2. Якщо користувач просить переглянути завдання:
    виклич list_tasks.

    3. Якщо користувач просить видалити завдання за описом:
    - ОБОВ'ЯЗКОВО виклич list_tasks.
    - Знайди завдання, опис якого відповідає запиту.
    - Скопіюй повний UUID цього завдання.
    - ОБОВ'ЯЗКОВО виклич delete_task з цим UUID.
    - Лише після успішного виклику delete_task
        повідом користувача про видалення.

    4. Якщо завдання не знайдено, повідом користувача.

    5. Якщо кілька завдань відповідають опису,
    попроси користувача уточнити, яке видалити.

    6. Не вигадуй ID.
    7. Не зупиняйся після list_tasks, якщо користувач
    просив видалити завдання.
    8. Не повідомляй про успішне видалення без
    підтвердження від delete_task.

"""



# агент з інструментами та store
agent = create_agent(
    model=model,
    tools=[add_task, list_tasks, delete_task],
    system_prompt=SYSTEM_PROMPT,
    store=store,
)


# тест агента
tests = [
    "Додай: купити хліб",
    "Додай: подзвонити лікарю",
    "Покажи всі завдання",
    "Видали завдання про хліб",
    "Що залишилось?"
]


for i, test in enumerate(tests, 1):
    print(f"\n{'='*50}")
    print(f"TEST {i}: {test}")
    print('='*50)

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": test}
        ]
    })

    print(response["messages"][-1].content)



TEST 1: Додай: купити хліб
Завдання "купити хліб" було успішно додано.

TEST 2: Додай: подзвонити лікарю
Завдання "подзвонити лікарю" було успішно додано.

TEST 3: Покажи всі завдання
Ось ваші завдання:

1. купити хліб (ID: c9fb28f2-31ca-4b5c-856f-c1bad57069d9)
2. подзвонити лікарю (ID: 840eddef-4f6b-45f7-a9ed-825418d7d22b)

TEST 4: Видали завдання про хліб
Завдання "купити хліб" було успішно видалено.

TEST 5: Що залишилось?
У вас залишилось одне завдання: подзвонити лікарю (ID: 840eddef-4f6b-45f7-a9ed-825418d7d22b).
